In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 18.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="ibm-granite/granite-3.3-2b-instruct"
)

messages = [
    {"role": "user", "content": "Who are you?"}
]
pipe(messages)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': 'I am an assistant designed to help answer your questions and provide information. I strive to provide concise and accurate responses in English.'}]}]

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-3.3-2b-instruct")
model = AutoModelForCausalLM. from_pretrained("ibm-granite/granite-3.3-2b-instruct")
messages = [
     {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
add_generation_prompt=True,
tokenize=True,
return_dict=True,
return_tensors="pt",
).to (model. device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

I am an assistant designed to help answer your questions and provide information. I strive to provide concise and accurate responses in English.<|end_of_text|>


In [5]:
!pip install -q gradio json5 jsonschema python-dotenv pydantic

import gradio as gr
import json
import json5
from jsonschema import ValidationError

# ==============================
# JSON Refiner Core
# ==============================

class JSONRefiner:

    def __init__(self, remove_null=False, sort_keys=False, indent=2, minify=False):
        self.remove_null = remove_null
        self.sort_keys = sort_keys
        self.indent = None if minify else indent

    def parse_json(self, text):

        if not text.strip():
            raise ValueError("Input JSON is empty")

        try:
            return json.loads(text)

        except Exception:
            try:
                return json5.loads(text)
            except Exception as e:
                raise ValueError(f"Invalid JSON: {str(e)}")

    def clean(self, obj):

        if isinstance(obj, list):
            return [self.clean(i) for i in obj]

        if isinstance(obj, dict):

            keys = sorted(obj.keys()) if self.sort_keys else obj.keys()
            new_obj = {}

            for k in keys:

                v = self.clean(obj[k])

                if self.remove_null and v is None:
                    continue

                new_obj[k] = v

            return new_obj

        return obj

    def refine(self, text):

        data = self.parse_json(text)
        data = self.clean(data)

        return json.dumps(data, indent=self.indent)


# ==============================
# Processing Function
# ==============================

def refine_json(input_json, remove_null, sort_keys, minify):

    try:

        refiner = JSONRefiner(
            remove_null=remove_null,
            sort_keys=sort_keys,
            minify=minify
        )

        result = refiner.refine(input_json)

        return "✅ JSON processed successfully", result

    except Exception as e:

        return f"❌ Error: {str(e)}", ""


# ==============================
# Gradio UI
# ==============================

with gr.Blocks(title="JSON Refiner Advanced") as app:

    gr.Markdown("## 🧰 JSON Refiner – Advanced Tool")

    with gr.Row():

        input_json = gr.Textbox(
            label="Input JSON",
            lines=15,
            placeholder="Paste your JSON here..."
        )

        output = gr.Textbox(
            label="Refined JSON",
            lines=15
        )

    with gr.Row():

        remove_null = gr.Checkbox(label="Remove Null Values")
        sort_keys = gr.Checkbox(label="Sort Keys Alphabetically")
        minify = gr.Checkbox(label="Minify JSON")

    run_btn = gr.Button("🚀 Refine JSON")

    status = gr.Textbox(label="Status")

    run_btn.click(
        refine_json,
        inputs=[input_json, remove_null, sort_keys, minify],
        outputs=[status, output]
    )

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ee7d362b1a7008d82f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
